# 5. Integração com CVAT

## Objetivo
- Remapear as coordenadas das *bounding boxes* (1024x1024) de volta para a **resolução original** dos DICOMs.
- Exportar as propostas como anotações em formato XML suportado nativamente pelo CVAT.
- Gerar as imagens brutas em formato PNG de alta resolução para simplificar o upload e evitar problemas de inversão de polaridade (`MONOCHROME1`) internos ao renderizador web do CVAT.
- Produzir dados verificáveis e prontos para a revisão manual de *candidatos_BAC*.

In [1]:
import os
import cv2
import pydicom
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from xml.dom import minidom
from pathlib import Path

## Carregamento de Dados e Fatores de Escala

In [2]:
# Carregar DataFrames
df_cand = pd.read_csv('../../data/interim/baseline_candidates.csv')
df_scales = pd.read_csv('../../data/interim/inbreast_batch1_scaling.csv')
df_manifest = pd.read_csv('../../data/interim/inbreast_batch1_ml_manifest.csv')

# Diretórios de saída CVAT
cvat_dir = Path('../../data/interim/cvat_export')
img_dir = cvat_dir / 'images'
img_dir.mkdir(parents=True, exist_ok=True)

## Exportando Imagens (PNG Alta Resolução)
Isso garante que a imagem submetida ao CVAT terá a polaridade corrigida e a resolução idêntica à do DICOM.

In [3]:
def export_dicom_to_png(dcm_path, out_path):
    ds = pydicom.dcmread(dcm_path)
    image = ds.pixel_array.astype(np.float32)
    if getattr(ds, 'PhotometricInterpretation', 'UNKNOWN') == 'MONOCHROME1':
        image = np.max(image) - image
    
    # Salvar em 16-bit para não perder faixa dinâmica se o CVAT suportar bem,
    # ou 8-bit com CLAHE leve/normalização. 
    # Optamos por 8-bit min-max simples para garantir compatibilidade universal.
    image_norm = (image - np.min(image)) / (np.max(image) - np.min(image) + 1e-8)
    image_8bit = (image_norm * 255).astype(np.uint8)
    
    cv2.imwrite(str(out_path), image_8bit)

# Gerar as imagens
image_name_map = {} # mapeia image_id para o nome do arquivo final gerado
for idx, row in df_manifest.iterrows():
    img_id = str(row['image_id'])
    dcm_path = row['dcm_path']
    png_name = f"{img_id}.png"
    out_path = img_dir / png_name
    
    export_dicom_to_png(dcm_path, out_path)
    image_name_map[int(img_id)] = png_name

print("Imagens originais convertidas para PNG exportadas em `cvat_export/images/`")

Imagens originais convertidas para PNG exportadas em `cvat_export/images/`


## Conversão de Coordenadas e Geração de CVAT XML 1.1

In [4]:
def prettify(elem):
    rough_string = ET.tostring(elem, 'utf-8')
    reparsed = minidom.parseString(rough_string)
    return reparsed.toprettyxml(indent="  ")

# Criar raiz do XML
annotations = ET.Element('annotations')
ET.SubElement(annotations, 'version').text = '1.1'
meta = ET.SubElement(annotations, 'meta')
task = ET.SubElement(meta, 'task')
ET.SubElement(task, 'name').text = 'INbreast_BAC_candidates'
ET.SubElement(task, 'size').text = str(len(df_manifest))
ET.SubElement(task, 'mode').text = 'annotation'

labels = ET.SubElement(task, 'labels')
label = ET.SubElement(labels, 'label')
ET.SubElement(label, 'name').text = 'candidato_BAC'
attributes = ET.SubElement(label, 'attributes')
ET.SubElement(attributes, 'attribute').text = 'score'
ET.SubElement(attributes, 'attribute').text = 'rank'

# Loop por imagem (Para ter o ID da imagem consistente no XML)
for img_idx, (_, row) in enumerate(df_manifest.iterrows()):
    img_id = int(row['image_id'])
    
    # Fatores de escala
    scale_info = df_scales[df_scales['image_id'] == img_id].iloc[0]
    scale_y = scale_info['scale_y']
    scale_x = scale_info['scale_x']
    orig_w = int(scale_info['original_width'])
    orig_h = int(scale_info['original_height'])
    
    # Elemento Imagem no XML
    image_elem = ET.SubElement(annotations, 'image', 
                               id=str(img_idx), 
                               name=image_name_map[img_id], 
                               width=str(orig_w), 
                               height=str(orig_h))
    
    # Puxar candidatos para esta imagem
    cands = df_cand[df_cand['image_id'] == img_id]
    
    for _, cand in cands.iterrows():
        # Parsing bbox
        bbox_str = cand['bbox'].strip("()").split(", ")
        min_r, min_c, max_r, max_c = map(int, bbox_str)
        
        # Inverter escala (1024 -> Original)
        # r = y (height), c = x (width)
        ytl = min_r / scale_y
        xtl = min_c / scale_x
        ybr = max_r / scale_y
        xbr = max_c / scale_x
        
        # Assegurar limites
        xtl = max(0.0, min(orig_w, xtl))
        ytl = max(0.0, min(orig_h, ytl))
        xbr = max(0.0, min(orig_w, xbr))
        ybr = max(0.0, min(orig_h, ybr))
        
        box = ET.SubElement(image_elem, 'box', 
                            label='candidato_BAC', 
                            occluded='0', 
                            xtl=f"{xtl:.2f}", 
                            ytl=f"{ytl:.2f}", 
                            xbr=f"{xbr:.2f}", 
                            ybr=f"{ybr:.2f}")
        
        score_attr = ET.SubElement(box, 'attribute', name='score')
        score_attr.text = f"{cand['score']:.3f}"
        
        rank_attr = ET.SubElement(box, 'attribute', name='rank')
        rank_attr.text = str(cand['rank'])

xml_str = prettify(annotations)
xml_path = cvat_dir / 'cvat_annotations.xml'
with open(xml_path, 'w', encoding='utf-8') as f:
    f.write(xml_str)

print(f"Exportação CVAT XML concluída: {xml_path}")


Exportação CVAT XML concluída: ../../data/interim/cvat_export/cvat_annotations.xml
